# The Composition Cliff

**A complete, reproducible study of when and why small transformers fail at function composition — and what actually fixes it.**

Everything here was measured on one CPU core. No GPU required. Full reproduction is ~40 minutes on a T4, ~2 hours on CPU.

---

## The three findings

**1. The composition cliff is real and generalizes.**
A fixed-depth transformer's ability to compose two functions collapses *sharply* above a threshold in the number of distinct functions. Not a gradual decay — a wall. Replicated in two structurally independent task families:

| task | algebra | 2nd step is | cliff location |
|---|---|---|---|
| S₅ state tracking | non-solvable group | a **lookup** | between domain 80 and 100 |
| Affine chain mod 17 | abelian ring | **arithmetic** | between domain 20 and 40 |

**2. Nothing you would normally reach for moves it.**
Fifteen interventions tested. All at chance.

**3. What fixes it is putting the intermediate value back into the token stream — and *only* that.**

This is the result. Two ways of supplying the model with the *identical, correct* intermediate value:

| method | S₅ | Affine |
|---|---|---|
| supervise it in the **hidden state** (aux head) | **1.0000** | **0.0645** (chance) |
| re-enter it as a **token** (chain of thought) | **1.0000** | **1.0000** |

On the affine task the aux head reaches **100% accuracy predicting the intermediate by step 1500**, while the final task never leaves chance. The model knows the number perfectly and cannot use it.

**The intermediate must be re-entered where attention can reach it. Having it present in the residual stream is not enough.**

---

## What this corrects

An earlier version of this work concluded that the mechanism was *intermediate supervision dose* — that ≥25% of examples needed correct intermediate labels. That reproduced cleanly on S₅ and **failed completely on the second task family.** It was a property of S₅'s lookup-shaped second step, not a general mechanism. The dose experiments are retained below because the failure is the informative part.

## 1 · Setup

In [ ]:
import sys, time, json, math, itertools
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337); np.random.seed(1337)
if DEV == "cpu":
    torch.set_num_threads(max(1, torch.get_num_threads()))
print("device:", DEV)
print("Every number in this notebook was produced on ONE CPU core.")

## 2 · Two task families

Both track a single state through *k* operations. They differ in what the second
step requires, which turns out to be the whole point.

**S₅ tracking** — state ∈ {0..4}, each op is a permutation. Composition is a
**lookup**: `PERM[g][p]`. S₅ is non-solvable, so its word problem is NC¹-complete;
a fixed-depth log-precision transformer is in uniform TC⁰ (Merrill & Sabharwal,
TACL 2023).

**Affine chain** — state ∈ ℤ₁₇, each op is `x → (a·x + b) mod 17`. Composition is
**arithmetic**: it requires multiplying a computed value by a parameter read from
a different token.

`domain` = how many distinct operations are drawn from. It is the knob that
produces the cliff in both.

In [ ]:
# ---------------- S5 state tracking ----------------
_P = list(itertools.permutations(range(5)))
PERM = np.array(_P, dtype=np.int64)              # PERM[g][p] = g(p)
N_PERM, N_POS = 120, 5
POFF, BOS, SEP, S5_VOCAB = 5, 125, 126, 127

class S5Track:
    def __init__(self, k=2, domain=120, seed=0):
        self.k, self.domain = k, domain
        self.rng = np.random.default_rng(seed)
    def sample(self, n):
        return (self.rng.integers(0,N_POS,size=n),
                self.rng.integers(0,self.domain,size=(n,self.k)))
    def trace(self, s, g):
        out=np.empty_like(g); cur=s.copy()
        for t in range(g.shape[1]):
            cur=PERM[g[:,t],cur]; out[:,t]=cur
        return out
    def batch_oneshot(self, n):
        s,g=self.sample(n); tr=self.trace(s,g)
        seq=np.concatenate([np.full((n,1),BOS),s[:,None],g+POFF,np.full((n,1),SEP)],axis=1)
        return (torch.from_numpy(seq).long().to(DEV),
                torch.from_numpy(tr[:,-1]).long().to(DEV),
                torch.from_numpy(tr[:,0]).long().to(DEV))
    def batch_cot(self, n):
        s,g=self.sample(n); tr=self.trace(s,g)
        seq=np.concatenate([np.full((n,1),BOS),s[:,None],g+POFF,
                            np.full((n,1),SEP),tr],axis=1)
        return (torch.from_numpy(seq[:,:-1]).long().to(DEV),
                torch.from_numpy(seq[:,1:]).long().to(DEV))
    VOCAB=S5_VOCAB
    n_states=N_POS
    @property
    def maxlen(self): return 2*self.k+4
    @property
    def chance(self): return 1.0/N_POS

# ---------------- Affine chain mod m ----------------
class AffineChain:
    def __init__(self, k=2, m=17, domain=120, seed=0):
        self.k,self.m=k,m; self.domain=domain
        self.rng=np.random.default_rng(seed)
        pairs=[]; a=1
        while len(pairs)<domain:
            for b in range(m):
                if len(pairs)>=domain: break
                if a%m!=0: pairs.append((a%m,b))
            a+=1
        self.pool=np.array(pairs[:domain],dtype=np.int64)
        self.BOS,self.SEP=m,m+1; self.OP_OFF=m+2
        self.VOCAB=self.OP_OFF+domain; self.n_states=m
    def sample(self,n):
        return (self.rng.integers(0,self.m,size=n),
                self.rng.integers(0,self.domain,size=(n,self.k)))
    def trace(self,x0,ops):
        out=np.empty_like(ops); cur=x0.copy()
        for t in range(ops.shape[1]):
            a=self.pool[ops[:,t],0]; b=self.pool[ops[:,t],1]
            cur=(a*cur+b)%self.m; out[:,t]=cur
        return out
    def batch_oneshot(self,n):
        x0,ops=self.sample(n); tr=self.trace(x0,ops)
        seq=np.concatenate([np.full((n,1),self.BOS),x0[:,None],ops+self.OP_OFF,
                            np.full((n,1),self.SEP)],axis=1)
        return (torch.from_numpy(seq).long().to(DEV),
                torch.from_numpy(tr[:,-1]).long().to(DEV),
                torch.from_numpy(tr[:,0]).long().to(DEV))
    def batch_cot(self,n):
        x0,ops=self.sample(n); tr=self.trace(x0,ops)
        seq=np.concatenate([np.full((n,1),self.BOS),x0[:,None],ops+self.OP_OFF,
                            np.full((n,1),self.SEP),tr],axis=1)
        return (torch.from_numpy(seq[:,:-1]).long().to(DEV),
                torch.from_numpy(seq[:,1:]).long().to(DEV))
    @property
    def maxlen(self): return self.k+3+self.k
    @property
    def chance(self): return 1.0/self.m

print("[ok] tasks")

## 3 · Model

One small decoder. Switches map to specific hypotheses: `n_layers` (parallel
depth), `n_loops` (recurrent depth — same weights reapplied), `aux_head` (a
second head that predicts the intermediate from the hidden state).

In [ ]:
def rope(x):
    B,H,T,D=x.shape; half=D//2
    f=1.0/(10000**(torch.arange(half,device=x.device).float()/half))
    a=torch.arange(T,device=x.device).float()[:,None]*f[None]
    cos,sin=a.cos(),a.sin(); x1,x2=x[...,:half],x[...,half:]
    return torch.cat([x1*cos-x2*sin, x1*sin+x2*cos],-1)

class Attn(nn.Module):
    def __init__(self,d,h,use_rope=False):
        super().__init__(); self.h,self.dh,self.use_rope=h,d//h,use_rope
        self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
    def forward(self,x,mask):
        B,T,D=x.shape
        q,k,v=[z.view(B,T,self.h,self.dh).transpose(1,2) for z in self.qkv(x).chunk(3,-1)]
        if self.use_rope: q,k=rope(q),rope(k)
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
        return self.o(y.transpose(1,2).reshape(B,T,D))

class Block(nn.Module):
    def __init__(self,d,h,ff=4,use_rope=False):
        super().__init__()
        self.n1=nn.LayerNorm(d); self.a=Attn(d,h,use_rope); self.n2=nn.LayerNorm(d)
        self.f=nn.Sequential(nn.Linear(d,ff*d),nn.GELU(),nn.Linear(ff*d,d))
    def forward(self,x,m): x=x+self.a(self.n1(x),m); return x+self.f(self.n2(x))

class LM(nn.Module):
    def __init__(self, vocab, d=64, layers=2, heads=4, maxlen=32, ff=4,
                 use_rope=False, use_pos=True, n_loops=1,
                 aux_head=False, n_aux=None):
        super().__init__()
        self.use_pos=use_pos and not use_rope; self.n_loops=n_loops
        self.tok=nn.Embedding(vocab,d)
        self.pos=nn.Embedding(maxlen,d) if self.use_pos else None
        self.blocks=nn.ModuleList([Block(d,heads,ff,use_rope) for _ in range(layers)])
        self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,vocab)
        self.aux=nn.Linear(d,n_aux) if aux_head else None
    def forward(self,idx):
        B,T=idx.shape; x=self.tok(idx)
        if self.pos is not None: x=x+self.pos(torch.arange(T,device=idx.device))[None]
        m=torch.triu(torch.full((T,T),float('-inf'),device=idx.device),1)
        for _ in range(self.n_loops):
            for b in self.blocks: x=b(x,m)
        x=self.norm(x)
        return self.head(x), (self.aux(x) if self.aux is not None else None)
    def n_params(self): return sum(p.numel() for p in self.parameters())

print("[ok] model")

## 4 · Training and evaluation

In [ ]:
STEPS, BS, LR = 2000, 256, 3e-3

def fit(task, steps=STEPS, d=64, layers=2, heads=4, ff=4, use_rope=False,
        use_pos=True, n_loops=1, lr=LR, cot=False, aux_frac=0.0,
        aux_kind="true", seed=0, schedule=None, bs=BS):
    """One trainer covering every arm in the study."""
    torch.manual_seed(seed); np.random.seed(seed)
    ns = task.n_states
    m = LM(task.VOCAB, d=d, layers=layers, heads=heads, maxlen=task.maxlen+2,
           ff=ff, use_rope=use_rope, use_pos=use_pos, n_loops=n_loops,
           aux_head=aux_frac>0, n_aux=ns).to(DEV)
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.01)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,lr,total_steps=steps)
    for s in range(steps):
        if cot:
            x,y=task.batch_cot(bs); lg,_=m(x)
            mask=torch.zeros_like(y,dtype=torch.bool); mask[:,-task.k:]=True
            loss=F.cross_entropy(lg[mask],y[mask])
        else:
            x,y,inter=task.batch_oneshot(bs); lg,aux=m(x)
            loss=F.cross_entropy(lg[:,-1,:ns],y)
            if aux_frac>0:
                tgt = inter if aux_kind=="true" else torch.randint(0,ns,(bs,),device=DEV)
                n=max(1,int(aux_frac*bs)); idx=torch.randperm(bs)[:n]
                loss=loss+F.cross_entropy(aux[idx,-1,:],tgt[idx])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step()
    return m

@torch.no_grad()
def acc_oneshot(m,task,n=4000,bs=1000):
    m.eval(); c=0; ns=task.n_states
    for _ in range(n//bs):
        x,y,_=task.batch_oneshot(bs); lg,_=m(x)
        c+=int((lg[:,-1,:ns].argmax(-1)==y).sum())
    m.train(); return c/(n//bs*bs)

@torch.no_grad()
def acc_cot(m,task,n=4000,bs=1000):
    """AUTOREGRESSIVE. Model writes its own chain; only the final state is scored."""
    m.eval(); c=0; ns=task.n_states
    for _ in range(n//bs):
        s,g=task.sample(bs); ans=task.trace(s,g)[:,-1]
        if isinstance(task,S5Track):
            pre=np.concatenate([np.full((bs,1),BOS),s[:,None],g+POFF,np.full((bs,1),SEP)],axis=1)
        else:
            pre=np.concatenate([np.full((bs,1),task.BOS),s[:,None],g+task.OP_OFF,
                                np.full((bs,1),task.SEP)],axis=1)
        cur=torch.from_numpy(pre).long().to(DEV)
        for _ in range(task.k):
            lg,_=m(cur); cur=torch.cat([cur,lg[:,-1,:ns].argmax(-1,keepdim=True)],1)
        c+=int((cur[:,-1].cpu().numpy()==ans).sum())
    m.train(); return c/(n//bs*bs)

print("[ok] harness")

## 5 · Sanity — the harness must succeed when there is no composition

k=1 is a pure lookup. If this is not ~1.0, everything downstream is meaningless.

In [ ]:
for name, T in [("S5", S5Track(k=1,domain=120)), ("Affine", AffineChain(k=1,domain=120))]:
    m=fit(T, steps=3000); a=acc_oneshot(m,T)
    print(f"  {name:7s} k=1  acc {a:.4f}  (chance {T.chance:.4f})  "
          f"{'OK' if a>0.9 else 'HARNESS BROKEN — STOP'}")

## 6 · Experiment 1 — the cliff, in both task families

Expected: perfect below a domain threshold, chance above it, with a sharp
transition. Measured locations: S₅ between 80 and 100; Affine between 20 and 40.

In [ ]:
print("S5   k=2, chance 0.2000")
for D_ in [5,20,60,80,100,120]:
    T=S5Track(2,domain=D_); m=fit(T); print(f"   domain {D_:4d}  acc {acc_oneshot(m,T):.4f}",flush=True)

print("\nAffine k=2, chance 0.0588")
for D_ in [5,20,40,80,120,200]:
    T=AffineChain(2,domain=D_); m=fit(T); print(f"   domain {D_:4d}  acc {acc_oneshot(m,T):.4f}",flush=True)

## 7 · Experiment 2 — the falsification battery

Fifteen attempts to move the cliff. Every one is a chance for the interpretation
to be wrong. **If any of these crosses, the conclusion changes** — which is the
point of running them.

In [ ]:
T=S5Track(2,domain=120)          # chance 0.20
print(f"Trying to break the cliff. chance = {T.chance}\n")
BATTERY=[
 ("baseline",                dict()),
 ("width d=128",             dict(d=128)),
 ("width d=256",             dict(d=256)),
 ("wide FFN 8x",             dict(ff=8)),
 ("heads 16",                dict(heads=16)),
 ("depth 4 layers",          dict(layers=4)),
 ("depth 6 layers",          dict(layers=6)),
 ("deep+narrow 8x32",        dict(layers=8,d=32)),
 ("recurrent depth x2",      dict(n_loops=2)),
 ("recurrent depth x4",      dict(n_loops=4)),
 ("rope",                    dict(use_rope=True)),
 ("no positional encoding",  dict(use_pos=False)),
 ("training 8000 steps",     dict(steps=8000)),
 ("training 20000 steps",    dict(steps=20000)),
 ("low lr 5e-4, 6000 steps", dict(lr=5e-4,steps=6000)),
]
for tag,kw in BATTERY:
    t0=time.time(); m=fit(T,**kw); a=acc_oneshot(m,T)
    flag="   <-- CLIFF CROSSED, CONCLUSION IS WRONG" if a>0.5 else ""
    print(f"  {tag:24s} acc {a:.4f}  params {m.n_params():>8,}  [{time.time()-t0:.0f}s]{flag}",flush=True)

## 8 · Experiment 3 — hidden-state supervision, and where it stops working

An auxiliary head predicts the intermediate value from the hidden state.

On **S₅** this works completely (1.0000) and shows a sharp dose threshold.
On **Affine** it fails flat — *even at 100% supervision*.

Three controls establish that on S₅ the effect is **information**, not parameters
or gradient flow: an unsupervised aux head, a random-target aux head, and a
structurally-wrong-target aux head all sit at chance.

In [ ]:
print("S5 (domain 120, chance 0.2000)")
T=S5Track(2,domain=120)
print("  controls:")
for tag,kw in [("aux head, no loss",  dict(aux_frac=0.0)),
               ("aux RANDOM target",  dict(aux_frac=1.0,aux_kind="random"))]:
    m=fit(T,**kw); print(f"    {tag:22s} acc {acc_oneshot(m,T):.4f}",flush=True)
print("  dose:")
for f in [0.01,0.05,0.10,0.15,0.25,0.50,1.00]:
    m=fit(T,aux_frac=f); print(f"    true target {f*100:5.1f}%      acc {acc_oneshot(m,T):.4f}",flush=True)

print("\nAffine (domain 120, chance 0.0588)")
T2=AffineChain(2,domain=120)
for f in [0.0,0.05,0.25,0.50,1.00]:
    m=fit(T2,aux_frac=f); print(f"    true target {f*100:5.1f}%      acc {acc_oneshot(m,T2):.4f}",flush=True)

### 8b · The diagnostic that explains the affine failure

Track the aux head's *own* accuracy alongside the final task. If the aux head
learns the intermediate perfectly while the task stays at chance, then having the
correct value in the hidden state is demonstrably **not sufficient**.

In [ ]:
T=AffineChain(2,domain=120); ns=T.n_states
torch.manual_seed(0)
m=LM(T.VOCAB,d=64,maxlen=T.maxlen+2,aux_head=True,n_aux=ns).to(DEV)
opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=0.01)
sch=torch.optim.lr_scheduler.OneCycleLR(opt,LR,total_steps=2000)
print(f"{'step':>6}{'final acc':>12}{'INTERMEDIATE acc':>20}   (chance {T.chance:.4f})")
for s in range(2000):
    x,y,inter=T.batch_oneshot(BS); lg,aux=m(x)
    (F.cross_entropy(lg[:,-1,:ns],y)+F.cross_entropy(aux[:,-1,:],inter)).backward()
    torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step(); opt.zero_grad()
    if (s+1)%500==0:
        m.eval()
        with torch.no_grad():
            xx,yy,ii=T.batch_oneshot(1000); l2,a2=m(xx)
            fa=(l2[:,-1,:ns].argmax(-1)==yy).float().mean().item()
            ia=(a2[:,-1,:].argmax(-1)==ii).float().mean().item()
        m.train(); print(f"{s+1:>6}{fa:>12.4f}{ia:>20.4f}",flush=True)

## 9 · Experiment 4 — the result

Identical information, two delivery routes. The only difference is whether the
intermediate lives in the **hidden state** or in the **token stream**.

CoT is evaluated autoregressively: the model generates its own chain and is
scored only on the final answer. It never sees a ground-truth intermediate at
test time.

In [ ]:
print(f"{'task':>10}{'hidden-state aux (100%)':>26}{'token-level CoT':>18}")
print("-"*54)
for name,T in [("S5", S5Track(2,domain=120)), ("Affine", AffineChain(2,domain=120))]:
    m_aux=fit(T,aux_frac=1.0);  a_aux=acc_oneshot(m_aux,T)
    m_cot=fit(T,cot=True);      a_cot=acc_cot(m_cot,T)
    print(f"{name:>10}{a_aux:>26.4f}{a_cot:>18.4f}   (chance {T.chance:.4f})",flush=True)

## 10 · Limitations

Stated plainly, because the study is only worth what its caveats allow.

- **Two task families, both synthetic.** The cliff generalizes across two
  algebras. It has *not* been shown on natural language, code, or any real
  workload. Do not assume the domain thresholds (80-100, 20-40) transfer.
- **k=2 only.** The composition depth is fixed at two throughout. Behaviour at
  larger k is unmeasured.
- **Small models, single seed per cell.** d=64, 2 layers, one seed. Differences
  under ~20% should not be read as meaningful; run-to-run noise at this scale is
  real (an earlier run had `cap=1` appear slower than `cap=3` purely by chance).
- **The falsification battery is not exhaustive.** Fifteen interventions failed.
  That is evidence that the cliff is robust, not proof that nothing can move it.
  A wider model, a different optimizer, or a curriculum not tried here might.
- **"Hidden state is insufficient" is demonstrated, not explained.** The aux head
  reaches 100% intermediate accuracy while the task stays at chance. *Why*
  attention can use a token but not an equally-present residual-stream value is
  not established by this study.
- **Prior art.** Chain-of-thought expressivity is established theory (Merrill &
  Sabharwal ICLR 2024; Li et al. ICLR 2024). Execution-trace and
  intermediate-step supervision are established practice (NExT, arXiv:2404.14662).
  The contribution here is the *controlled comparison* of delivery route at
  matched information, and the cliff's replication across two algebras — not the
  idea that intermediates help.

## 11 · Reproducing

Run cells 1-4 once, then any experiment cell independently. Cell 5 (sanity) should
be run first — if k=1 is not near 1.0, the harness is broken and nothing else
means anything. Cell 7 is the one that can prove the conclusion wrong; it is the
most valuable cell in the notebook.